# Parameter sensitivity analysis

How sensitive is the along-track velocity RMSE to the drifter model
parameters $k_d$ (drogue drag coefficient) and $\tilde{m}_d$ (drogue
added mass)?

We perform a 2D grid search over ($k_d$, $\tilde{m}_d$) by integrating
the full drifter ODE at each observed position using
`DroguedDrifter.get_final_drift_batch` and evaluating the velocity RMSE
against observed drifter motion.

## Imports

In [ ]:
import copernicusmarine as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

## Parameters

In [ ]:
CSV_PATH = "examples/baltic_drifters/data/drifters_clean.csv"
TIME_START = "2023-04-24"
TIME_END = "2023-05-10"
RESAMPLE_INTERVAL = "1h"
K_D_DEFAULT = 154.0   # kg/m (Callies et al. geometry)
M_TILDE_D_DEFAULT = 101.0  # kg (added mass)

## Load observed trajectories

Resample to hourly fixes and compute observed velocity from finite
differences. Same preprocessing as notebook 10.

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=["date_UTC"])

records = []
for drifter_id, grp in df.groupby("D_number"):
    grp = grp.set_index("date_UTC").sort_index()
    grp_h = grp.resample(RESAMPLE_INTERVAL).first().dropna(subset=["Latitude"])
    grp_h["D_number"] = drifter_id
    records.append(grp_h.reset_index())

df_h = pd.concat(records, ignore_index=True)

obs_records = []
for drifter_id, grp in df_h.groupby("D_number"):
    grp = grp.sort_values("date_UTC").reset_index(drop=True)
    dt = grp["date_UTC"].diff().dt.total_seconds()
    dlat = grp["Latitude"].diff()
    dlon = grp["Longitude"].diff()
    lat_rad = np.radians(grp["Latitude"])

    grp["u_obs"] = dlon * 1852 * 60 * np.cos(lat_rad) / dt
    grp["v_obs"] = dlat * 1852 * 60 / dt
    obs_records.append(grp.iloc[1:])

df_obs = pd.concat(obs_records, ignore_index=True)
print(f"{len(df_obs)} hourly fixes across {df_obs['D_number'].nunique()} drifters")

## Sample CMEMS effective currents along tracks

Interpolate Eulerian currents (surface and 3 m) and Stokes drift along
each drifter position and time. Same approach as notebook 10.

In [ ]:
TIME = slice(TIME_START, TIME_END)

lon_min, lon_max = df_obs["Longitude"].min() - 0.2, df_obs["Longitude"].max() + 0.2
lat_min, lat_max = df_obs["Latitude"].min() - 0.2, df_obs["Latitude"].max() + 0.2
LON = slice(lon_min, lon_max)
LAT = slice(lat_min, lat_max)

ds_phy = cm.open_dataset(
    dataset_id="cmems_mod_bal_phy_anfc_PT1H-i",
    service="arco-geo-series",
)[["uo", "vo"]].sel(
    longitude=LON, latitude=LAT, time=TIME, depth=slice(0, 5),
)

WAVE_VARS = [
    "VHM0_WW", "VTM01_WW", "VMDR_WW",
    "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",
    "VHM0_SW2", "VTM01_SW2", "VMDR_SW2",
]

ds_wav = cm.open_dataset(
    dataset_id="cmems_mod_bal_wav_anfc_PT1H-i",
    service="arco-geo-series",
).sel(longitude=LON, latitude=LAT, time=TIME)[WAVE_VARS]

print(f"Physics: {ds_phy.dims}")
print(f"Waves:   {ds_wav.dims}")

In [ ]:
times = xr.DataArray(pd.DatetimeIndex(df_obs["date_UTC"].values), dims="obs")
lons = xr.DataArray(df_obs["Longitude"].values, dims="obs")
lats = xr.DataArray(df_obs["Latitude"].values, dims="obs")

depth_levels = ds_phy.depth.values
z_surf = depth_levels[0]
z_3m = depth_levels[3]

euler_sampled = ds_phy.interp(
    longitude=lons, latitude=lats, time=times, method="linear",
).compute()

df_obs["uo_surf"] = euler_sampled["uo"].sel(depth=z_surf).values
df_obs["vo_surf"] = euler_sampled["vo"].sel(depth=z_surf).values
df_obs["uo_3m"] = euler_sampled["uo"].sel(depth=z_3m).values
df_obs["vo_3m"] = euler_sampled["vo"].sel(depth=z_3m).values

print(f"Eulerian sampled: {df_obs[['uo_surf','vo_surf']].notna().all(axis=1).sum()} valid of {len(df_obs)}")

### Stokes drift at surface and drogue depth

In [ ]:
g = 9.81

PARTITIONS = [
    ("VHM0_WW", "VTM01_WW", "VMDR_WW"),
    ("VHM0_SW1", "VTM01_SW1", "VMDR_SW1"),
    ("VHM0_SW2", "VTM01_SW2", "VMDR_SW2"),
]

wav_sampled = ds_wav.interp(
    longitude=lons, latitude=lats, time=times, method="linear",
).compute()

for label, z in [("0m", 0.0), ("3m", float(z_3m))]:
    u_st = np.zeros(len(df_obs))
    v_st = np.zeros(len(df_obs))

    for hs_var, t_var, dir_var in PARTITIONS:
        hs = wav_sampled[hs_var].values
        T = wav_sampled[t_var].values
        dir_from = wav_sampled[dir_var].values

        valid = (hs > 0.01) & np.isfinite(T) & (T > 0.1)
        A = np.where(valid, hs / 2, 0.0)
        sigma = np.where(valid, 2 * np.pi / np.where(valid, T, 1.0), 0.0)
        k = sigma**2 / g
        theta = np.deg2rad(270.0 - np.where(valid, dir_from, 0.0))
        decay = np.exp(-2 * k * z)
        mag = A**2 * sigma * k * decay

        u_st += mag * np.cos(theta)
        v_st += mag * np.sin(theta)

    df_obs[f"u_stokes_{label}"] = u_st
    df_obs[f"v_stokes_{label}"] = v_st

print(f"Stokes at surface: mean speed {np.sqrt(df_obs['u_stokes_0m']**2 + df_obs['v_stokes_0m']**2).mean():.4f} m/s")
print(f"Stokes at 3m:      mean speed {np.sqrt(df_obs['u_stokes_3m']**2 + df_obs['v_stokes_3m']**2).mean():.4f} m/s")

### Effective currents and valid subset

In [ ]:
df_obs["u_eff_surf"] = df_obs["uo_surf"] + df_obs["u_stokes_0m"]
df_obs["v_eff_surf"] = df_obs["vo_surf"] + df_obs["v_stokes_0m"]
df_obs["u_eff_3m"] = df_obs["uo_3m"] + df_obs["u_stokes_3m"]
df_obs["v_eff_3m"] = df_obs["vo_3m"] + df_obs["v_stokes_3m"]

df_valid = df_obs.dropna(subset=["uo_surf", "uo_3m"]).copy()
print(f"{len(df_valid)} valid fixes out of {len(df_obs)} ({len(df_obs) - len(df_valid)} dropped near coast)")

## 2D sensitivity: $k_d$ vs $\tilde{m}_d$ via ODE integration

Integrate the full drifter ODE at each observed position with varying
($k_d$, $\tilde{m}_d$) pairs using `DroguedDrifter.get_final_drift_batch`.

We use a coarse 8x8 grid since each evaluation requires solving the ODE
system to steady state for all valid fixes.

In [ ]:
from drogued_drifters.drifter import DroguedDrifter

N_GRID = 8
k_d_2d = np.linspace(50, 300, N_GRID)
m_tilde_d_2d = np.linspace(20, 200, N_GRID)

# Extract arrays from valid subset
u_obs = df_valid["u_obs"].values
v_obs = df_valid["v_obs"].values
U_b = df_valid["u_eff_surf"].values.copy()
V_b = df_valid["v_eff_surf"].values.copy()
U_d = df_valid["u_eff_3m"].values.copy()
V_d = df_valid["v_eff_3m"].values.copy()

rmse_2d = np.empty((N_GRID, N_GRID))

for i, k_d in enumerate(k_d_2d):
    for j, m_td in enumerate(m_tilde_d_2d):
        dd = DroguedDrifter(k_d=k_d, m_tilde_d=m_td)
        xd_drift, yd_drift, _, _ = dd.get_final_drift_batch(
            U_b=U_b, V_b=V_b, U_d=U_d, V_d=V_d,
        )
        rmse_2d[i, j] = np.sqrt(np.mean((u_obs - xd_drift)**2 + (v_obs - yd_drift)**2))
    print(f"  k_d = {k_d:.0f} done")

print("2D grid search complete.")

### RMSE heatmap

In [ ]:
da_rmse = xr.DataArray(
    rmse_2d,
    dims=["k_d", "m_tilde_d"],
    coords={"k_d": k_d_2d, "m_tilde_d": m_tilde_d_2d},
    attrs={"long_name": "Velocity RMSE", "units": "m/s"},
)
da_rmse.coords["k_d"].attrs = {"long_name": "$k_d$", "units": "kg/m"}
da_rmse.coords["m_tilde_d"].attrs = {"long_name": r"$\tilde{m}_d$", "units": "kg"}

da_rmse.plot()
plt.plot(M_TILDE_D_DEFAULT, K_D_DEFAULT, "w*", markersize=15, markeredgecolor="k", label="default")
plt.legend()
plt.show()

# Print min
i_min, j_min = np.unravel_index(np.argmin(rmse_2d), rmse_2d.shape)
print(f"2D minimum: k_d = {k_d_2d[i_min]:.0f} kg/m, m_tilde_d = {m_tilde_d_2d[j_min]:.0f} kg, RMSE = {rmse_2d[i_min, j_min]:.4f} m/s")